In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score

In [2]:
data = pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/pope_base_des_05_11_2025.csv")

In [3]:
preds = []
for i in data["answer"]:
    pred = i[:10].lower()
    if "yes" in pred:
        preds.append("yes")
    elif "no" in pred:
        preds.append("no")
    else:
        preds.append("unknown")
        
data["prediciton"] = preds
pd.Series(preds).value_counts()

yes    4817
no     4093
Name: count, dtype: int64

In [4]:
data.head(2)

,question,gt_answer,question_id,image_id,image_path,data_type,answer,prediciton
0,Is there a snowboard in the image?,yes,3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"Yes, there is a snowboard in the image, and th...",yes
1,Is there a backpack in the image?,no,14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,adversarial,"No, there is no backpack in the image. The per...",no


In [5]:
for name, group in data.groupby("data_type"):
    print(f"Type: {name}")
    # f1_scores = f1_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    acc_scores = accuracy_score(group["gt_answer"].tolist(), group["prediciton"].tolist())
    # print(f"F1 Score: {f1_scores}")
    print(f"Accuracy Score: {acc_scores}")

Type: adversarial
Accuracy Score: 0.7953333333333333
Type: popular
Accuracy Score: 0.8586666666666667
Type: random
Accuracy Score: 0.8917525773195877


In [6]:
accuracy_score(data["gt_answer"], preds)

0.8481481481481481

In [7]:
# tatget-word based evaluation

# tatget-word based evaluation

In [8]:
result_df = pd.read_pickle("/Data2/Arun-UAV/NLP/vision_halu/total_flow_testing_results/pope/pope_llava_label_with_evidence_and_attn_detection_05_11_2025.pkl")

In [12]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack


In [11]:
result_df["target_word"] = result_df["question"].apply(lambda x: x.replace("Is there a", "").replace("in the image?","").strip())

In [14]:
all_preds = []
for ans in result_df["answer"]:
    pred = ans[:10].lower()
    if "yes" in pred:
        all_preds.append("yes")
    elif "no" in pred:
        all_preds.append("no")
    else:
        all_preds.append("unknown")

In [16]:
pd.Series(all_preds).value_counts()
result_df["pred_label"] = all_preds

In [17]:
result_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word,pred_label
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard,yes
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack,no


In [175]:
failed_df = result_df[result_df["pred_label"] == result_df["gt_answer"]]

In [176]:
failed_df.head(2)

,question,answer,question_id,image_id,image_path,gt_answer,data_type,labels_with_evidence,target_word,pred_label
0,Is there a snowboard in the image?,"Yes, there is a snowboard in the image, and th...",3f4a998f-caf1-428d-baa3-80c014d751e2,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,yes,adversarial,"[{'word': 'Yes,', 'evidence': [0.023237491, 0....",snowboard,yes
1,Is there a backpack in the image?,"No, there is no backpack in the image. The per...",14371c3a-6a62-419f-893c-1ca360490a7b,310196,/Data2/Arun-UAV/NLP/vision_halu/benchmarks/coc...,no,adversarial,"[{'word': 'No,', 'evidence': [0.007417327, 0.0...",backpack,no


In [177]:
failed_df.index = range(len(failed_df))

In [203]:
import torch

faild = []
corrrected_label = []

both_halu = 0
detect_halu = 0
evi_halu = 0
d_res = []
d_res_pred = []
for inx, row in failed_df.iterrows():
    try:
        pred = row["pred_label"]
        t_word = row["target_word"].split(" ")[-1].strip()
        evi_df = pd.DataFrame(row["labels_with_evidence"])
        req_info = evi_df[evi_df["word"] == t_word].iloc[0]
        prob = req_info["label"].item()
        img_evi = (torch.tensor(req_info["evidence"]) >= 0.4).int().sum().item()
        
        if prob <= 0.5 and img_evi <= 1:
            corrrected_label.append("no")
            d_res.append(pred)
            d_res_pred.append("no")
        elif prob > 0.75 and img_evi <= 2:
            new_img_evi = (torch.tensor(req_info["evidence"]) >= 0.3).int().sum().item()
            if new_img_evi > 2:
                corrrected_label.append("yes")
                # d_res_pred.append("yes")
            else:
                corrrected_label.append("no")
                # d_res_pred.append("no")
            # d_res.append(pred)
        elif prob <=  0.75 and img_evi > 2:
            corrrected_label.append("yes")
            # d_res.append(pred)
            # d_res_pred.append("yes")
        else:
            # d_res.append(pred)
            # d_res_pred.append("yes")
            corrrected_label.append("yes")

    except Exception as e:
        faild.append(inx)
        corrrected_label.append(pred)

In [204]:
accuracy_score(failed_df["gt_answer"].tolist(), corrrected_label)

0.7500330819108112

In [205]:
pd.Series(d_res).value_counts()

yes    1393
no        2
Name: count, dtype: int64

In [206]:
pd.Series(d_res_pred).value_counts()

no    1395
Name: count, dtype: int64

In [217]:
df["image_id"].nunique()

9929

In [220]:
df_1["image_id"].nunique()

11312

In [221]:
vqa = pd.concat([df_2, df_3])

In [223]:
vqa["image_id"].nunique()

14804

In [215]:
pd.read_csv("/Data2/Arun-UAV/NLP/vision_halu/haloc/haloc_extension/caption/original_sampled_14k.csv")

,Unnamed: 0,image_id,prompt,hallucinated_text,source_text,source_metadata,qa_metadata,qa_ids,annotations,id,split
0,19046,2382873,<image>Analyze the image in a comprehensive an...,Bottom left side of the image there is a traff...,Bottom left side of the image there is a traff...,"{'source': 'localized_narratives', 'id': 'sp_2...",[],['04256137'],"{'object': [{'obj': {'name': 'bus', 'char_inde...",caption_23545,train
1,7985,2368415,<image>Can you elaborate on the elements of th...,There are two boats on a water. On the right s...,There are two boats on a water. On the right s...,"{'source': 'localized_narratives', 'id': 'sp_4...",[],"['051055498', '051055532']","{'object': [{'obj': {'name': 'bench', 'char_in...",caption_9807,train
2,20055,2393446,<image>What is this photo about?,In the foreground a tower visible and a window...,In the foreground a tower visible and a window...,"{'source': 'localized_narratives', 'id': 'sp_5...",[],['061055637'],"{'object': [{'obj': {'name': 'clouds', 'char_i...",caption_24806,train
3,21684,2328300,<image>Describe the following image.,In this image I can see two person are riding ...,In this image I can see two person are riding ...,"{'source': 'localized_narratives', 'id': 'sp_5...",[],['0930967'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_26797,train
4,20706,2363891,<image>Describe the following image.,A bathtub sits next to a white wall. There is ...,A bathtub sits next to a white wall. There is ...,"{'source': 'stanford', 'id': 'sp_30166'}",[],['07277178'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_25579,train
...,...,...,...,...,...,...,...,...,...,...,...
13995,6358,2397248,<image>Describe the following image.,"In this image, in the left side there is a gir...","In this image, in the left side there is a gir...","{'source': 'localized_narratives', 'id': 'sp_1...",[],"['03900126', '03900163']","{'object': [], 'attribute': [{'attribute': {'n...",caption_7710,train
13996,3847,2341206,<image>Analyze the image in a comprehensive an...,In the image there is a lady with spectacles i...,In the image there is a lady with spectacles i...,"{'source': 'localized_narratives', 'id': 'sp_2...",[],['1248031'],"{'object': [{'obj': {'name': 'remote control',...",caption_4652,train
13997,28614,2413496,<image>What do you think is going on in this s...,In this picture we can see a group of animals ...,In this picture we can see a group of animals ...,"{'source': 'localized_narratives', 'id': 'sp_5...",[],['c_15927562'],"{'object': [], 'attribute': [{'attribute': {'n...",caption_35305,train
13998,15026,2408106,<image>Can you elaborate on the elements of th...,In this image I can see a boy standing in maro...,In this image I can see a boy standing in gras...,"{'source': 'localized_narratives', 'id': 'sp_1...",[],"['c_1722851', '1722747']","{'object': [{'obj': {'name': 'frisbee', 'char_...",caption_18586,train
